In [1]:
import numpy as np
from scipy.optimize import fsolve
from scipy.integrate import quad
import scipy.stats as stats
from tqdm import tqdm
import os, sys
_EXP_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _EXP_ROOT not in sys.path:
    sys.path.insert(0, _EXP_ROOT)
from transrr_lib.robust_ridge_optimizer import solve_robust_ridge
from risk_solver import solve_gaussian
from joblib import Parallel, delayed

In [2]:
def simu_gaussian(k_idx, p, n, beta_0, sigma, w_hat, delta, eta, tau):

    np.random.seed(k_idx)

    XX = np.random.normal(size=(n, p))
    YY = XX @ beta_0 + np.random.normal(0, sigma, n)

    Y_adjusted = YY - (XX @ w_hat) # 注意：这里是 train_X @ w_hat_trans
    initial_guess2 = np.linalg.solve(
        XX.T @ XX / len(YY) + tau * np.eye(p),
        XX.T @ Y_adjusted / len(YY) # Y_adjusted 是 (train_y - train_X @ w_hat_trans)
    )
    delta_hat = solve_robust_ridge(XX, Y_adjusted, tau, delta, eta, initial_beta=initial_guess2)

    err_norm = np.sum((delta_hat + w_hat - beta_0) ** 2)

    # print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, var_errnorm: {np.var(errnorm):.3e}, err_norm2: {np.mean(errnorm):.3e}, r2: {r2:.3e}")

    return err_norm

In [3]:
p_range = [200, 400, 800]
n_range = [200, 400, 800]
tau = 1
tau1 = 1
sigma = 1
sigma1 = 2
delta = 1.35
eta = 0.1
K = 1000

alpha = 0.05

results = np.zeros((3, K + 2))
for i in range(3):
    p = p_range[i]
    n = n_range[i]
    nn = n*2
    rng = np.random.RandomState(1)
    beta_0 = rng.uniform(size=p)
    beta_0 = beta_0 / np.sqrt(n)
    w_0 = rng.uniform(size=p)
    w_0 = w_0 / np.sqrt(n)

    delta_0 = beta_0 - w_0

    kappa = p // n

    XX1 = rng.multivariate_normal(np.zeros(p), np.eye(p), nn)
    YY1 = XX1 @ w_0 + rng.normal(0, sigma1, nn)

    initial_guess = np.linalg.solve(XX1.T @ XX1 / nn + tau1 * np.eye(p), XX1.T @ YY1) / nn
    w_hat = solve_robust_ridge(XX1, YY1, tau1, delta, eta, initial_beta=initial_guess)

    err_norm = Parallel(n_jobs=-1)(
        delayed(simu_gaussian)(i, p, n, beta_0, sigma, w_hat, delta, eta, tau)
        for i in tqdm(range(K), desc="Progress")
    )

    c, r2 = solve_gaussian(sigma, delta, eta, kappa, tau, beta_0, w_hat)

    print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, sd_errnorm: {np.std(err_norm):.4f}, err_norm2: {np.mean(err_norm):.4f}, r2: {r2:.4f}")

    # 将 errnorm 的每个元素存储在矩阵中
    results[i, :K] = err_norm
    results[i, K] = c
    results[i, K + 1] = r2



Progress: 100%|██████████| 1000/1000 [00:00<00:00, 1203.30it/s]


p:200, n:200, tau:1, simu_times:1000, sd_errnorm: 0.0318, err_norm2: 0.3653, r2: 0.3649


Progress: 100%|██████████| 1000/1000 [00:01<00:00, 851.45it/s]


p:400, n:400, tau:1, simu_times:1000, sd_errnorm: 0.0208, err_norm2: 0.3472, r2: 0.3477


Progress: 100%|██████████| 1000/1000 [00:05<00:00, 177.69it/s]


p:800, n:800, tau:1, simu_times:1000, sd_errnorm: 0.0151, err_norm2: 0.3603, r2: 0.3598


In [4]:
csv_filename = f'res/res_kappa{kappa}_tau{tau}_gaussian.csv'

np.savetxt(csv_filename, results, delimiter=",", header=",".join([f"errnorm_{i}" for i in range(K)] + ["c", "r2"]), comments="")

In [5]:
p_range = [200, 400, 800]
n_range = [50, 100, 200]
tau = 1
tau1 = 1
sigma = 1
sigma1 = 2
delta = 1.35
eta = 0.1
K = 1000

alpha = 0.05

results = np.zeros((3, K + 2))
for i in range(3):
    p = p_range[i]
    n = n_range[i]
    nn = n*2
    rng = np.random.RandomState(1)
    beta_0 = rng.uniform(size=p)
    beta_0 = beta_0 / np.sqrt(n)
    w_0 = rng.uniform(size=p)
    w_0 = w_0 / np.sqrt(n)

    delta_0 = beta_0 - w_0

    kappa = p // n

    XX1 = rng.multivariate_normal(np.zeros(p), np.eye(p), nn)
    YY1 = XX1 @ w_0 + rng.normal(0, sigma1, nn)

    initial_guess = np.linalg.solve(XX1.T @ XX1 / nn + tau1 * np.eye(p), XX1.T @ YY1) / nn
    w_hat = solve_robust_ridge(XX1, YY1, tau1, delta, eta, initial_beta=initial_guess)

    err_norm = Parallel(n_jobs=-1)(
        delayed(simu_gaussian)(i, p, n, beta_0, sigma, w_hat, delta, eta, tau)
        for i in tqdm(range(K), desc="Progress")
    )

    c, r2 = solve_gaussian(sigma, delta, eta, kappa, tau, beta_0, w_hat)

    print(f"p:{p}, n:{n}, tau:{tau}, simu_times:{K}, sd_errnorm: {np.std(err_norm):.4f}, err_norm2: {np.mean(err_norm):.4f}, r2: {r2:.4f}")

    # 将 errnorm 的每个元素存储在矩阵中
    results[i, :K] = err_norm
    results[i, K] = c
    results[i, K + 1] = r2


Progress: 100%|██████████| 1000/1000 [00:00<00:00, 5513.34it/s]


p:200, n:50, tau:1, simu_times:1000, sd_errnorm: 0.0734, err_norm2: 1.3415, r2: 1.3419


Progress: 100%|██████████| 1000/1000 [00:00<00:00, 3110.33it/s]


p:400, n:100, tau:1, simu_times:1000, sd_errnorm: 0.0531, err_norm2: 1.3565, r2: 1.3544


Progress: 100%|██████████| 1000/1000 [00:01<00:00, 622.02it/s]


p:800, n:200, tau:1, simu_times:1000, sd_errnorm: 0.0427, err_norm2: 1.5261, r2: 1.5247


In [6]:
csv_filename = f'res/res_kappa{kappa}_tau{tau}_gaussian.csv'

np.savetxt(csv_filename, results, delimiter=",", header=",".join([f"errnorm_{i}" for i in range(K)] + ["c", "r2"]), comments="")